<a href="https://colab.research.google.com/github/mayayonathan/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-09%20%E2%80%94%20Public%20API%20to%20Clean%20Table%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [2]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    last = None
    for attempt in range(tries):
        try:
            r = requests.get(BASE, params=params, timeout=15)
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            if r.status_code not in RETRY_STATUSES:
                raise RuntimeError(f'{BASE} returned {r.status_code}: {r.text[:200]}')
            last = f'status {r.status_code}'
        except requests.RequestException as e:
            last = repr(e)
        delay = 1 * (2 ** attempt)
        print(f'attempt {attempt + 1} failed ({last}); retrying in {delay}s')
        time.sleep(delay)
    raise RuntimeError(f'gave up after {tries} attempts: {last}')

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [3]:
used_fallback = False

try:
    raw = get_weather('2026-09-05', '2026-10-03')
    raw = raw[raw['time'].isin(['2026-09-05', '2026-10-03'])].reset_index(drop=True)
    print('pulled from Open-Meteo:')
    print(raw)
except Exception as e:
    print('live pull failed, will use fallback:', e)
    used_fallback = True

pulled from Open-Meteo:
         time  temperature_2m_max  precipitation_sum
0  2026-09-05                29.4                2.5
1  2026-10-03                22.0               14.7


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [4]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

if used_fallback:
    wx = FALLBACK.copy()
else:
    wx = raw.copy()

wx['source'] = 'fallback sample' if used_fallback else 'open-meteo'
wx

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [5]:
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})[['date', 'temp_max', 'precip_mm', 'source']]

wx['date'] = pd.to_datetime(wx['date'])
wx = wx.sort_values('date').reset_index(drop=True)
wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
1,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [6]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [7]:
wettest = wx.loc[wx['precip_mm'].idxmax()]
driest = wx.loc[wx['precip_mm'].idxmin()]
diff = wettest['precip_mm'] - driest['precip_mm']

print(
    f"{wettest['date'].date()} was the wetter of the two home weekends, "
    f"with {wettest['precip_mm']:.1f}mm of rain versus {driest['precip_mm']:.1f}mm "
    f"on {driest['date'].date()} -- a difference of {diff:.1f}mm."
)

2026-10-03 was the wetter of the two home weekends, with 14.7mm of rain versus 2.5mm on 2026-09-05 -- a difference of 12.2mm.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [8]:
from datetime import datetime

print('Source:    Open-Meteo historical archive API')
print('Endpoint: ', BASE)
print('Coordinates: lat=38.03, lon=-78.51 (Charlottesville, VA)')
print('Date range:', wx['date'].min().date(), 'to', wx['date'].max().date())
print('Pulled:   ', datetime.now().strftime('%Y-%m-%d %H:%M'))
print('Data is:  ', 'fallback sample -- not a real pull' if used_fallback else 'real, pulled live from Open-Meteo')

Source:    Open-Meteo historical archive API
Endpoint:  https://archive-api.open-meteo.com/v1/archive
Coordinates: lat=38.03, lon=-78.51 (Charlottesville, VA)
Date range: 2026-09-05 to 2026-10-03
Pulled:    2026-10-11 19:12
Data is:   real, pulled live from Open-Meteo


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

The merging onto 200 days of sales would silently discard every row for the days the retrieve failed if `get_weather` provided an empty DataFrame rather than raising; alternatively, depending on the join type, those rows with `NaN` rain would be regarded as "no rain" rather than "unknown." Low or no rain would be correlated with income on certain days, according to the chart, which appears to be a true finding: "rain doesn't move sales much." No one would notice it because there are no errors or missing-data warnings in the output, simply a chart that tells a more subdued story than the actual weather would have. From the outside, the bug and the business conclusion appear to be similar. The difference would only be apparent to someone who occurred to compare the row count to the initial 200 days or who was previously aware that those weekends were rainy.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [9]:
cities = {'Charlottesville': (38.03, -78.51), 'Orlando': (28.54, -81.38)}

frames = []
for city, (lat, lon) in cities.items():
    try:
        c_raw = get_weather('2026-09-05', '2026-10-03', lat=lat, lon=lon)
        c_raw = c_raw[c_raw['time'].isin(['2026-09-05', '2026-10-03'])].reset_index(drop=True)
        c_used_fallback = False
    except Exception as e:
        print(f'{city}: live pull failed ({e}), using fallback')
        c_raw = FALLBACK.copy()
        c_used_fallback = True
    c_raw['source'] = 'fallback sample' if c_used_fallback else 'open-meteo'
    c_raw['city'] = city
    frames.append(c_raw)

by_city = pd.concat(frames, ignore_index=True).rename(columns={
    'time': 'date', 'temperature_2m_max': 'temp_max', 'precipitation_sum': 'precip_mm',
})[['city', 'date', 'temp_max', 'precip_mm', 'source']]
by_city['date'] = pd.to_datetime(by_city['date'])
by_city

,city,date,temp_max,precip_mm,source
0,Charlottesville,2026-09-05,29.4,2.5,open-meteo
1,Charlottesville,2026-10-03,22.0,14.7,open-meteo
2,Orlando,2026-09-05,32.1,3.9,open-meteo
3,Orlando,2026-10-03,30.4,14.5,open-meteo
